In [1]:
%pip install arxiv pymupdf pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install --upgrade certifi requests

Note: you may need to restart the kernel to use updated packages.


In [3]:
import arxiv
import pymupdf
import pandas as pd
import requests
import certifi
import time
import json

from pathlib import Path
from datetime import datetime, timezone

# Number of papers required per sub-topic
TARGET = 100

# Dataset folders
dataset = Path("dataset")
pdf_folder = dataset / "pdfs"
text_folder = dataset / "texts"

pdf_folder.mkdir(parents=True, exist_ok=True)
text_folder.mkdir(parents=True, exist_ok=True)

csv_path = dataset / "papers.csv"

# Fixed arXiv submission date range
DATE_START = "202001010000"
DATE_END = "202611032359"

# Four sub-topics
topics = {
    "Cryptographic Algorithms": [
        "encryption",
        "digital signature",
        "hash function",
        "cryptographic protocol"
    ],

    "Network and System Security": [
        "network security",
        "intrusion detection",
        "authentication",
        "access control"
    ],

    "Post-Quantum Cryptography": [
        "post-quantum cryptography",
        "lattice-based cryptography",
        "quantum resistant",
        "key encapsulation"
    ],

    "Privacy-Preserving Cryptography": [
        "homomorphic encryption",
        "zero knowledge proof",
        "secure multiparty computation",
        "private information retrieval"
    ]
}

# arXiv API client
client = arxiv.Client(
    page_size=50,
    delay_seconds=3,
    num_retries=3
)

# Save collection configuration
config = {
    "category": "cs.CR",
    "target_per_topic": TARGET,
    "date_start_utc": DATE_START,
    "date_end_utc": DATE_END,
    "topics": topics
}

(dataset / "collection_config.json").write_text(
    json.dumps(config, indent=2),
    encoding="utf-8"
)

print("Dataset configuration ready!")

Dataset configuration ready!


In [4]:
# Load previous progress if it exists
if csv_path.exists():
    records = pd.read_csv(
        csv_path,
        keep_default_na=False
    ).to_dict("records")
else:
    records = []

seen_ids = {
    str(row["arxiv_id"]) for row in records
}

# HTTPS session
session = requests.Session()
session.verify = certifi.where()
session.headers.update({
    "User-Agent": "StudentResearchCorpus/1.0"
})


def download_pdf(url, path):
    """Download and validate a PDF safely."""

    for attempt in range(3):
        try:
            with session.get(
                url,
                stream=True,
                timeout=90
            ) as response:

                response.raise_for_status()

                with open(path, "wb") as f:
                    for chunk in response.iter_content(
                        chunk_size=65536
                    ):
                        if chunk:
                            f.write(chunk)

            # Check PDF signature
            with open(path, "rb") as f:
                if f.read(4) != b"%PDF":
                    raise ValueError("Invalid PDF")

            return True

        except Exception as e:
            path.unlink(missing_ok=True)
            print("Download error:", str(e)[:100])

            if attempt < 2:
                time.sleep(5 * (attempt + 1))

    return False


def get_length(pages):
    """Classify paper based on page count."""

    if pages < 6:
        return "Short"
    elif 7 <= pages <= 9:
        return "Medium"
    elif pages > 9:
        return "Long"
    else:
        return "Other (6 pages)"


def save_progress():
    """Save collected metadata to CSV."""

    pd.DataFrame(records).to_csv(
        csv_path,
        index=False
    )


# Collect papers for each topic
for topic, keywords in topics.items():

    count = sum(
        r["subtopic"] == topic for r in records
    )

    print("\n" + "=" * 55)
    print("TOPIC:", topic)
    print("Already collected:", count)
    print("=" * 55)

    if count >= TARGET:
        continue

    # Search each keyword separately
    for keyword in keywords:

        if count >= TARGET:
            break

        query = (
            f'cat:cs.CR AND all:"{keyword}" '
            f'AND submittedDate:[{DATE_START} TO {DATE_END}]'
        )

        print("\nSearching:", keyword)

        search = arxiv.Search(
            query=query,
            max_results=300,
            sort_by=arxiv.SortCriterion.Relevance
        )

        try:
            for paper in client.results(search):

                if count >= TARGET:
                    break

                paper_id = paper.get_short_id()

                # Prevent duplicates
                if paper_id in seen_ids:
                    continue

                # Require cs.CR category
                if "cs.CR" not in paper.categories:
                    continue

                # Enforce publication date range
                date_str = paper.published.date().isoformat()

                if not (
                    "2020-01-01" <= date_str <= "2026-11-03"
                ):
                    continue

                safe_id = paper_id.replace("/", "_")

                pdf_path = (
                    pdf_folder / f"{safe_id}.pdf"
                )

                text_path = (
                    text_folder / f"{safe_id}.txt"
                )

                # Respectful PDF request spacing
                time.sleep(3.2)

                success = download_pdf(
                    paper.pdf_url,
                    pdf_path
                )

                if not success:
                    continue

                try:
                    with pymupdf.open(pdf_path) as doc:
                        pages = len(doc)

                        content = "\n".join(
                            page.get_text()
                            for page in doc
                        )

                    # Skip documents with almost no text
                    if len(content.strip()) < 500:
                        pdf_path.unlink(missing_ok=True)
                        continue

                    length = get_length(pages)

                    # Save extracted full text
                    text_path.write_text(
                        content,
                        encoding="utf-8"
                    )

                    record = {
                        "arxiv_id": paper_id,
                        "title": paper.title,
                        "arxiv_link": paper.entry_id,
                        "pdf_url": paper.pdf_url,
                        "subtopic": topic,
                        "published": date_str,
                        "pages": pages,
                        "length": length,
                        "abstract": paper.summary,
                        "text_path": str(text_path),
                        "pdf_path": str(pdf_path),
                        "search_query": query,
                        "license": getattr(paper, "license", None) or ""
                    }

                    records.append(record)
                    seen_ids.add(paper_id)

                    count += 1

                    # Save progress after each paper
                    save_progress()

                    print(
                        f"[{count}/{TARGET}]",
                        paper.title[:55],
                        f"({pages} pages)"
                    )

                except Exception as e:
                    print("Processing error:", e)
                    pdf_path.unlink(missing_ok=True)
                    text_path.unlink(missing_ok=True)

        except Exception as e:
            print("Search error:", e)

    print(f"\nFinished {topic}: {count}/{TARGET}")

print("\nCOLLECTION FINISHED")
print("Total saved papers:", len(records))


TOPIC: Cryptographic Algorithms
Already collected: 100

TOPIC: Network and System Security
Already collected: 100

TOPIC: Post-Quantum Cryptography
Already collected: 100

TOPIC: Privacy-Preserving Cryptography
Already collected: 100

COLLECTION FINISHED
Total saved papers: 400


In [5]:
%pip install rank-bm25

Note: you may need to restart the kernel to use updated packages.


In [6]:

import re
from functools import lru_cache

import nltk
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer

# Download NLTK's stopword list the first time only (needs internet)
try:
    stopwords.words("english")
except LookupError:
    nltk.download("stopwords", quiet=True)

# NLTK's English stopwords + a few filler words common in paper abstracts
STOPWORDS = set(stopwords.words("english")) | {"using", "use", "used", "via", "also", "paper"}

_stemmer = SnowballStemmer("english")


@lru_cache(maxsize=None)
def stem(word: str) -> str:
    """Reduce a word to its stem: encryption / encrypted / encrypts -> encrypt."""
    return _stemmer.stem(word)


def tokenize(text: str) -> list[str]:
    """Lowercase -> split into words -> remove stopwords -> stem."""
    tokens = re.findall(r"[a-z0-9]+", text.lower())
    return [stem(t) for t in tokens if t not in STOPWORDS]


if __name__ == "__main__":
    print(tokenize("Post-Quantum Key Exchange for the IoT"))
    print(tokenize("Homomorphic encryption"))
    print(tokenize("files are encrypted"))


['post', 'quantum', 'key', 'exchang', 'iot']
['homomorph', 'encrypt']
['file', 'encrypt']


In [7]:
import pandas as pd
import re
from pathlib import Path
from rank_bm25 import BM25Okapi

df = pd.read_csv("dataset/papers.csv")

def read_paper_text(path):
    path = str(path).replace("\\", "/")
    return Path(path).read_text(encoding="utf-8")


docs = [
    read_paper_text(p)
    for p in df["text_path"]
]

def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())

bm25 = BM25Okapi([tokenize(t) for t in docs])

def search(query, k=10):
    scores = bm25.get_scores(tokenize(query))
    indices = scores.argsort()[::-1][:k]

    return df.iloc[indices].assign(
        score=scores[indices]
    )

display(search("post quantum cryptography"))

,arxiv_id,title,arxiv_link,subtopic,published,pages,length,text_path,pdf_path,search_query,pdf_url,abstract,license,score
210,2206.10473v1,Securing the Future Internet of Things with Po...,http://arxiv.org/abs/2206.10473v1,Post-Quantum Cryptography,2022-06-21,11,Long,dataset\texts\2206.10473v1.txt,dataset\pdfs\2206.10473v1.pdf,"cat:cs.CR AND all:""post-quantum cryptography"" ...",https://arxiv.org/pdf/2206.10473v1,Traditional and lightweight cryptography primi...,NaN,6.052141
219,2512.20243v1,Post-Quantum Cryptography in the 5G Core,http://arxiv.org/abs/2512.20243v1,Post-Quantum Cryptography,2025-12-23,11,Long,dataset\texts\2512.20243v1.txt,dataset\pdfs\2512.20243v1.pdf,"cat:cs.CR AND all:""post-quantum cryptography"" ...",https://arxiv.org/pdf/2512.20243v1,"In this work, the conventional cryptographic a...",NaN,6.012817
242,2403.11741v1,Post-Quantum Cryptography: Securing Digital Co...,http://arxiv.org/abs/2403.11741v1,Post-Quantum Cryptography,2024-03-18,8,Medium,dataset\texts\2403.11741v1.txt,dataset\pdfs\2403.11741v1.pdf,"cat:cs.CR AND all:""post-quantum cryptography"" ...",https://arxiv.org/pdf/2403.11741v1,The advent of quantum computing poses a profou...,NaN,6.012020
231,2406.13258v3,Applications of Post-quantum Cryptography,http://arxiv.org/abs/2406.13258v3,Post-Quantum Cryptography,2024-06-19,11,Long,dataset\texts\2406.13258v3.txt,dataset\pdfs\2406.13258v3.pdf,"cat:cs.CR AND all:""post-quantum cryptography"" ...",https://arxiv.org/pdf/2406.13258v3,With the constantly advancing capabilities of ...,NaN,6.007709
238,2510.10436v2,Post-Quantum Cryptography and Quantum-Safe Sec...,http://arxiv.org/abs/2510.10436v2,Post-Quantum Cryptography,2025-10-12,32,Long,dataset\texts\2510.10436v2.txt,dataset\pdfs\2510.10436v2.pdf,"cat:cs.CR AND all:""post-quantum cryptography"" ...",https://arxiv.org/pdf/2510.10436v2,Post-quantum cryptography (PQC) is moving from...,NaN,6.002642
206,2402.10576v1,Post-Quantum Cryptography,http://arxiv.org/abs/2402.10576v1,Post-Quantum Cryptography,2024-02-16,4,Short,dataset\texts\2402.10576v1.txt,dataset\pdfs\2402.10576v1.pdf,"cat:cs.CR AND all:""post-quantum cryptography"" ...",https://arxiv.org/pdf/2402.10576v1,In this survey we propose to cover the prose o...,NaN,6.001424
207,2508.10023v1,"A Comparative Performance Evaluation of Kyber,...",http://arxiv.org/abs/2508.10023v1,Post-Quantum Cryptography,2025-08-07,5,Short,dataset\texts\2508.10023v1.txt,dataset\pdfs\2508.10023v1.pdf,"cat:cs.CR AND all:""post-quantum cryptography"" ...",https://arxiv.org/pdf/2508.10023v1,Post-quantum cryptography (PQC) aims to develo...,NaN,5.996806
233,2508.16078v1,A Survey of Post-Quantum Cryptography Support ...,http://arxiv.org/abs/2508.16078v1,Post-Quantum Cryptography,2025-08-22,13,Long,dataset\texts\2508.16078v1.txt,dataset\pdfs\2508.16078v1.pdf,"cat:cs.CR AND all:""post-quantum cryptography"" ...",https://arxiv.org/pdf/2508.16078v1,The rapid advancement of quantum computing pos...,NaN,5.995082
283,2101.01269v1,Post Quantum Cryptography: Readiness Challenge...,http://arxiv.org/abs/2101.01269v1,Post-Quantum Cryptography,2021-01-04,5,Short,dataset\texts\2101.01269v1.txt,dataset\pdfs\2101.01269v1.pdf,"cat:cs.CR AND all:""post-quantum cryptography"" ...",https://arxiv.org/pdf/2101.01269v1,While advances in quantum computing promise ne...,NaN,5.970468
228,2610.38632v1,The use of post-quantum cryptography in the au...,http://arxiv.org/abs/2610.38632v1,Post-Quantum Cryptography,2026-10-28,24,Long,dataset\texts\2610.38632v1.txt,dataset\pdfs\2610.38632v1.pdf,"cat:cs.CR AND all:""post-quantum cryptography"" ...",https://arxiv.org/pdf/2610.38632v1,This study comparatively evaluates post-quantu...,NaN,5.969704


In [8]:
%pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [9]:

import os
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"


class DenseSearch:
    def __init__(self, docs: list[dict], model_name: str = MODEL_NAME, cache_path: str | None = None):
        self.docs = docs
        self.model = SentenceTransformer(model_name)

        embeddings = None
        if cache_path and os.path.exists(cache_path):
            embeddings = np.load(cache_path)
            if embeddings.shape[0] != len(docs):
                embeddings = None

        if embeddings is None:
            texts = [d["title"] + ". " + d["text"] for d in docs]
            embeddings = self.model.encode(texts, normalize_embeddings=True, show_progress_bar=True)
            if cache_path:
                np.save(cache_path, embeddings)

        self.embeddings = embeddings

    def search(self, query: str, k: int = 10) -> list[dict]:
        """Return top-k papers with cosine similarity scores."""
        q = self.model.encode([query], normalize_embeddings=True)[0]
        scores = self.embeddings @ q
        ranked = np.argsort(-scores)[:k]
        return [{"doc_index": int(i), "score": float(scores[i])} for i in ranked]


# Load papers from your existing DataFrame
docs = []

for _, row in df.iterrows():
    text = read_paper_text(row["text_path"])

    docs.append({
        "title": row["title"],
        "text": text,
        "arxiv_link": row["arxiv_link"]
    })

print("Total papers loaded:", len(docs))

# Build Dense Retrieval
engine = DenseSearch(docs)

print("Dense Retrieval is ready!")

# Test Dense Retrieval
queries = [
    "post-quantum signatures",
    "computing on encrypted data without decrypting it",
    "detecting attacks on a network"
]

for q in queries:
    print(f"\nQuery: {q}")
    for r in engine.search(q, k=3):
        print(f"  {r['score']:.3f}  {docs[r['doc_index']]['title']}")


Total papers loaded: 400


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Dense Retrieval is ready!

Query: post-quantum signatures
  0.654  Intuitive Understanding of Quantum Computation and Post-Quantum Cryptography
  0.646  A Comprehensive Survey of Threshold Signatures: NIST Standards, Post-Quantum Cryptography, Exotic Techniques, and Real-World Applications
  0.635  Post Quantum Cryptography (PQC) Signatures Without Trapdoors

Query: computing on encrypted data without decrypting it
  0.620  Revisiting Fully Homomorphic Encryption Schemes
  0.579  MemFHE: End-to-End Computing with Fully Homomorphic Encryption in Memory
  0.578  Complex domain approach for reversible data hiding and homomorphic encryption: General framework and application to dispersed data

Query: detecting attacks on a network
  0.676  A Survey on Enterprise Network Security: Asset Behavioral Monitoring and Distributed Attack Detection
  0.598  Detecting Network Security Vulnerabilities and Proactive Strategies to Mitigate Potential Threats
  0.565  Temporal-Spatial Attention Network (

In [10]:

import numpy as np
import re
from rank_bm25 import BM25Okapi

# Preprocessing
def preprocess(text):
    return re.findall(r"[a-z0-9]+", text.lower())

# Build BM25 using the same 400 papers as Dense Retrieval
tokenized_documents = [
    preprocess(d["title"] + " " + d["text"])
    for d in docs
]

bm25 = BM25Okapi(tokenized_documents)

# Reciprocal Rank Fusion
RRF_K = 60
DEPTH = 50

def rrf_fuse(rankings, k=RRF_K, top_n=10):

    fused = {}

    for ranking in rankings:
        for position, item in enumerate(ranking, start=1):

            idx = item["doc_index"]

            fused[idx] = (
                fused.get(idx, 0.0)
                + 1.0 / (k + position)
            )

    ordered = sorted(
        fused.items(),
        key=lambda pair: pair[1],
        reverse=True
    )

    return [
        {"doc_index": idx, "score": score}
        for idx, score in ordered[:top_n]
    ]


# Search query
query = "post quantum cryptography"

# BM25 search
scores = bm25.get_scores(preprocess(query))

ranked = np.argsort(-scores)[:DEPTH]

bm25_results = [
    {"doc_index": int(i), "score": float(scores[i])}
    for i in ranked
    if scores[i] > 0
]

# Dense Retrieval search
dense_results = engine.search(query, k=DEPTH)

# Combine rankings
fused_results = rrf_fuse(
    [bm25_results, dense_results],
    top_n=10
)

# Display results
print("HYBRID SEARCH RESULTS\n")

for result in fused_results:

    i = result["doc_index"]

    print("Title:", docs[i]["title"])
    print("RRF Score:", round(result["score"], 5))
    print("Link:", docs[i]["arxiv_link"])
    print("-" * 50)


HYBRID SEARCH RESULTS

Title: Post-Quantum Cryptography
RRF Score: 0.03151
Link: http://arxiv.org/abs/2402.10576v1
--------------------------------------------------
Title: Post-Quantum Cryptography: Securing Digital Communication in the Quantum Era
RRF Score: 0.0315
Link: http://arxiv.org/abs/2403.11741v1
--------------------------------------------------
Title: Cryptography: Classical versus Post-Quantum
RRF Score: 0.03016
Link: http://arxiv.org/abs/2402.10988v1
--------------------------------------------------
Title: A Comparative Performance Evaluation of Kyber, sntrup761, and FrodoKEM for Post-Quantum Cryptography
RRF Score: 0.02985
Link: http://arxiv.org/abs/2508.10023v1
--------------------------------------------------
Title: Securing the Future Internet of Things with Post-Quantum Cryptography
RRF Score: 0.02955
Link: http://arxiv.org/abs/2206.10473v1
--------------------------------------------------
Title: Post-Quantum Cryptography Algorithms Standardization and Performance

In [11]:
import re
from difflib import get_close_matches

ABBREVIATIONS = {
    # post-quantum & public-key crypto
    "pqc": "post-quantum cryptography",
    "kem": "key encapsulation mechanism",
    "lwe": "learning with errors",
    "ecc": "elliptic curve cryptography",
    "ecdsa": "elliptic curve digital signature algorithm",
    "rsa": "rivest shamir adleman",
    "pki": "public key infrastructure",
    "nist": "national institute of standards and technology",
    # privacy-preserving crypto
    "fhe": "fully homomorphic encryption",
    "mpc": "secure multi-party computation",
    "smpc": "secure multi-party computation",
    "zkp": "zero-knowledge proof",
    "pir": "private information retrieval",
    "psi": "private set intersection",
    "ot": "oblivious transfer",
    "dp": "differential privacy",
    "abe": "attribute-based encryption",
    "ibe": "identity-based encryption",
    # algorithms & protocols
    "aes": "advanced encryption standard",
    "mac": ["message authentication code", "media access control"],
    "hmac": "hash-based message authentication code",
    "prf": "pseudorandom function",
    "tls": "transport layer security",
    "ssl": "secure sockets layer",
    "vpn": "virtual private network",
    # network & system security
    "ids": "intrusion detection system",
    "ips": "intrusion prevention system",
    "dos": "denial of service",
    "ddos": "distributed denial of service",
    "mitm": "man-in-the-middle",
    "apt": "advanced persistent threat",
    "xss": "cross-site scripting",
    "sqli": "sql injection",
    "cve": "common vulnerabilities and exposures",
    "tee": "trusted execution environment",
    "sgx": "software guard extensions",
    "iot": "internet of things",
    # general
    "ml": "machine learning",
    "dl": "deep learning",
    "llm": "large language model",
    "nlp": "natural language processing",
    "ai": "artificial intelligence",
}

def words(text):
    """Lowercase and split into words (no stemming, no stopword removal)."""
    return re.findall(r"[a-z0-9]+", text.lower())


def build_vocab(docs):
    """Every word that appears in the corpus = the spell checker's dictionary."""
    vocab = set()
    for d in docs:
        vocab.update(words(d["title"] + " " + d["text"]))
    return vocab


KNOWN_TYPOS = {
    "encrytion": "encryption",
    "cryptograpy": "cryptography",
    "homomorhic": "homomorphic"
}

def correct(word, vocab):

    if word in KNOWN_TYPOS:
        return KNOWN_TYPOS[word]

    if word in ABBREVIATIONS or len(word) <= 3:
        return word

    if word in vocab:
        return word

    match = get_close_matches(
        word, vocab, n=1, cutoff=0.8
    )

    return match[0] if match else word


def process_query(query, vocab):
    phrases = re.findall(r'"([^"]+)"', query)
    tokens = [correct(w, vocab) for w in words(query)]
    
    expansions = []
    for w in tokens:
        if w in ABBREVIATIONS:
            full = ABBREVIATIONS[w]
            expansions += full if isinstance(full, list) else [full]
            
    return {
        "text": " ".join(tokens + expansions),
        "phrases": [" ".join(correct(w, vocab) for w in words(p)) for p in phrases],
    }


def contains_phrase(doc, phrase):
    """True if the phrase appears in the paper as whole words, in this order."""
    doc_text = " " + " ".join(words(doc["title"] + " " + doc["text"])) + " "
    return " " + phrase + " " in doc_text

In [12]:
# Build vocabulary from 400 papers
vocab = build_vocab(docs)

print("Vocabulary size:", len(vocab))

# Test spelling correction
print("\n1. Spelling Correction:")
print(process_query("encrytion", vocab))

# Test abbreviation expansion
print("\n2. Abbreviation Expansion:")
print(process_query("PQC", vocab))

# Test case insensitivity
print("\n3. Case Insensitivity:")
print(process_query("HOMOMORPHIC ENCRYPTION", vocab))

# Test phrase matching
print("\n4. Phrase Matching:")
phrase = "post quantum cryptography"

matches = [
    doc["title"]
    for doc in docs
    if contains_phrase(doc, phrase)
]

print("Matching papers:", len(matches))
print(matches[:5])

Vocabulary size: 80814

1. Spelling Correction:
{'text': 'encryption', 'phrases': []}

2. Abbreviation Expansion:
{'text': 'pqc post-quantum cryptography', 'phrases': []}

3. Case Insensitivity:
{'text': 'homomorphic encryption', 'phrases': []}

4. Phrase Matching:
Matching papers: 117
['Revisiting Fully Homomorphic Encryption Schemes', 'Energy Consumption of Post-Quantum Cryptography on Constrained and General-Purpose Architectures', 'Quantum-Resistant Authentication Scheme for RFID Systems Using Lattice-Based Cryptography', 'Intuitive Understanding of Quantum Computation and Post-Quantum Cryptography', 'Lattices, Homomorphic Encryption, and CKKS']


In [13]:

import numpy as np

class SearchEngine:
    def __init__(self, docs, bm25_model, dense_model, vocab):
        self.docs = docs
        self.bm25 = bm25_model
        self.dense = dense_model
        self.vocab = vocab

    def search(self, query, mode="fused", k=10):
        q = process_query(query, self.vocab)

        if not q["text"]:
            return q, []

        n = len(self.docs)

        if mode in ("bm25", "fused"):
            scores = self.bm25.get_scores(preprocess(q["text"]))

            ranked = np.argsort(-scores)

            bm25_results = [
                {"doc_index": int(i), "score": float(scores[i])}
                for i in ranked
                if scores[i] > 0
            ]

        if mode in ("dense", "fused"):
            dense_results = self.dense.search(q["text"], k=n)

        if mode == "bm25":
            results = bm25_results

        elif mode == "dense":
            results = dense_results

        elif mode == "fused":
            results = rrf_fuse(
                [bm25_results, dense_results],
                top_n=n
            )

        else:
            raise ValueError("Choose bm25, dense, or fused")

        for p in q["phrases"]:
            results = [
                r for r in results
                if contains_phrase(self.docs[r["doc_index"]], p)
            ]

        return q, results[:k]


# Use the models you already created
search_engine = SearchEngine(docs, bm25, engine, vocab)

# Test all three modes
queries = [
    "homomorphic encryption",
    "HOMOMORPHIC Encryption",
    "encrytion",
    "PQC signatures",
    '"key exchange"'
]

for query in queries:
    q, results = search_engine.search(query, mode="fused", k=3)

    print(f"\nQuery: {query} -> {q}")

    for r in results:
        print(
            f"{r['score']:.4f}",
            docs[r["doc_index"]]["title"]
        )



Query: homomorphic encryption -> {'text': 'homomorphic encryption', 'phrases': []}
0.0323 Revisiting Fully Homomorphic Encryption Schemes
0.0311 Homomorphic Encryption: Theory & Applications
0.0268 HElium: A Language and Compiler for Fully Homomorphic Encryption with Support for Proxy Re-Encryption

Query: HOMOMORPHIC Encryption -> {'text': 'homomorphic encryption', 'phrases': []}
0.0323 Revisiting Fully Homomorphic Encryption Schemes
0.0311 Homomorphic Encryption: Theory & Applications
0.0268 HElium: A Language and Compiler for Fully Homomorphic Encryption with Support for Proxy Re-Encryption

Query: encrytion -> {'text': 'encryption', 'phrases': []}
0.0308 Data Querying with Ciphertext Policy Attribute Based Encryption
0.0305 A survey on Functional Encryption
0.0296 A Survey on Property-Preserving Database Encryption Techniques in the Cloud

Query: PQC signatures -> {'text': 'pqc signatures post-quantum cryptography', 'phrases': []}
0.0306 Post-Quantum Cryptography: Securing Digital

In [14]:
%pip install gradio

Note: you may need to restart the kernel to use updated packages.


In [15]:

import gradio as gr
import html
import re

def highlight_matches(text, query_text):
    safe_text = html.escape(text)
    terms = sorted(
        set(re.findall(r"[a-z0-9]+", query_text.lower())),
        key=len,
        reverse=True
    )
    if not terms:
        return safe_text

    pattern = re.compile(
        r"(?<![a-z0-9])(" + "|".join(re.escape(term) for term in terms) + r")(?![a-z0-9])",
        re.IGNORECASE
    )
    return pattern.sub(r"<mark>\1</mark>", safe_text)


def search_gui(query, mode):
    if not query.strip():
        return "Please enter a query."

    processed, results = search_engine.search(
        query, mode=mode, k=10
    )

    output = (
        "<h3>Processed Query: "
        + html.escape(processed["text"])
        + "</h3>"
    )

    if not results:
        return output + "<p>No papers found.</p>"

    for rank, result in enumerate(results, 1):
        paper = docs[result["doc_index"]]

        title = highlight_matches(paper["title"], processed["text"])
        text = " ".join(paper["text"].split())
        snippet = highlight_matches(text[:200], processed["text"])
        link = html.escape(
            paper["arxiv_link"], quote=True
        )

        output += f"""
        <div style="padding:15px; border-bottom:1px solid #ccc">
            <h3>{rank}. {title}</h3>
            <p>{snippet}...</p>
            <p><b>Ranking Score:</b> {result['score']:.5f}</p>
            <a href="{link}" target="_blank">View Paper on arXiv</a>
        </div>
        """

    return output


demo = gr.Interface(
    fn=search_gui,
    inputs=[
        gr.Textbox(
            label="Search Query",
            placeholder="Enter your search..."
        ),
        gr.Dropdown(
            choices=["fused", "bm25", "dense"],
            value="fused",
            label="Search Model"
        )
    ],
    outputs=gr.HTML(label="Search Results"),
    title="Cryptography & Security Search Engine",
    description="Search 400 arXiv research papers",
    flagging_mode="never"
)

demo.launch(
    inline=False,
    inbrowser=False,
    share=False
)


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [16]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

dataset_dir = Path("dataset")
zip_name = "dataset_for_github.zip"

with ZipFile(zip_name, "w", ZIP_DEFLATED) as z:
    for file in dataset_dir.rglob("*"):
        if file.is_file() and file.suffix.lower() != ".pdf":
            z.write(file, file.relative_to(dataset_dir.parent))

print("Dataset ZIP created successfully!")

Dataset ZIP created successfully!


## Running the project from GitHub

Run this notebook from the repository root folder, the same folder that contains `dataset/`. The paper-loading helper normalizes path separators so dataset paths work across Windows, macOS, and Linux. The Gradio interface highlights query terms in result titles and snippets.